**Librerias**

In [ ]:
!pip install bptk-py
!pip install pandas
!pip install matplotlib

In [ ]:
import BPTK_Py
from BPTK_Py import Model
from BPTK_Py import sd_functions as sd
import numpy as np, pandas as pd, matplotlib.pyplot as plt

**Diccionario con parametros** Falta cambiar a los valores de los parametros reales basados en la literatura. Son tasas anuales (trim:0.25) hace que sean trimestrales.

In [ ]:
PARAMETROS = {
    #definir tiempo
    "t_inicio": 0.0, "t_fin": 10.0, "trim": 0.25,

    "N_pob_inicial": 100.0,
    "N_nacimiento": 0.40, "N_muerte_natural": 0.40,
    #se puso la tasa de muertes para simular las muertes naturales para poder ver la dinamica del sistema real sin contemplar estas
    "N_eficacia_polin": 1.0,

    "competencia_recursos": 0.004,
    #compentencia de las las berries por el polen.
    # donde sacar el dato (es importante ya que influye en la eficiencia del cultivo)

    "C_renta_inicial": 0.0,
    "C_tasa_contratacion": 1.0, "C_tasa_retiro": 1.0,
    "C_eficacia_polin": 1.5,
    #la eficacia de polinizacion es mayor de las colmenas porque hay mayor concentración de población

    "polinizacion_requerida": 150.0,
    "rendimiento_maximo": 20.0,
    "t_rendimiento": 0.02,
#t_rendimiento es cuanto tiempo toma para tener cierto rendimiento
}
#los que empiezan con N son nativos
#los que empiezan con C son colmenas
#falta agregar las unidades de todos los parametros

**Definir tiempo**

In [ ]:
def crear_modelo(p, nombre="PolinizadoresBerries"):
    return Model(starttime=p["t_inicio"], stoptime=p["t_fin"], trim=p["trim"], name=nombre)
    #pusimos todo en los parametros para poder cambiar las fechas en un solo lugar sin tener que cambiar el codigo

**polinizadores bucles e interacciones**

In [ ]:
def polinizadores(model, p):
    v = {}

    # Niveles
    v["poblacion_nativa"]  = model.stock("poblacion_nativa")
    v["colmenas_rentadas"] = model.stock("colmenas_rentadas")

    # Flujos
    for f in ["tasa_regeneracion_nativa", "tasa_mortalidad_nativa",
              "tasa_alquiler_introduccion", "tasa_retiro_comercial"]:
        v[f] = model.flow(f)

    # Constantes
    for c in ["N_nacimiento", "N_muerte_natural", "competencia_recursos",
              "N_eficacia_polin", "C_tasa_contratacion", "C_tasa_retiro",
              "C_eficacia_polin", "polinizacion_requerida"]:
        v[c] = model.constant(c); v[c].equation = p[c]

    # Auxiliares
    for a in ["polinizacion_nativa_disponible", "polinizacion_rentada",
              "polinizacion_efectiva_total", "deficit_polinizacion", "efecto_competencia"]:
        v[a] = model.converter(a)

    v["poblacion_nativa"].initial_value  = p["N_pob_inicial"]
    v["colmenas_rentadas"].initial_value = p["C_renta_inicial"]

    v["polinizacion_nativa_disponible"].equation = v["poblacion_nativa"] * v["N_eficacia_polin"]
    v["polinizacion_rentada"].equation = v["colmenas_rentadas"] * v["C_eficacia_polin"]
    v["polinizacion_efectiva_total"].equation = v["polinizacion_nativa_disponible"] + v["polinizacion_rentada"]
    v["deficit_polinizacion"].equation = sd.max(0.0, v["polinizacion_requerida"] - v["polinizacion_efectiva_total"])
    v["efecto_competencia"].equation = v["colmenas_rentadas"] * v["competencia_recursos"]

    v["tasa_regeneracion_nativa"].equation = v["poblacion_nativa"] * v["N_nacimiento"]
    v["tasa_mortalidad_nativa"].equation = v["poblacion_nativa"] * (v["N_muerte_natural"] + v["efecto_competencia"])
    v["tasa_alquiler_introduccion"].equation = v["deficit_polinizacion"] * v["C_tasa_contratacion"]
    v["tasa_retiro_comercial"].equation = v["colmenas_rentadas"] * v["C_tasa_retiro"]

    v["poblacion_nativa"].equation  = v["tasa_regeneracion_nativa"] - v["tasa_mortalidad_nativa"]
    v["colmenas_rentadas"].equation = v["tasa_alquiler_introduccion"] - v["tasa_retiro_comercial"]
    return v

In [ ]:
def modulo_agricola(model, p, v):
    v["rendimiento_maximo"] = model.constant("rendimiento_maximo"); v["rendimiento_maximo"].equation = p["rendimiento_maximo"]
    v["t_rendimiento"] = model.constant("t_rendimiento"); v["t_rendimiento"].equation = p["t_rendimiento"]
    v["rendimiento_agricola"] = model.converter("rendimiento_agricola")
    v["rendimiento_agricola"].equation = v["rendimiento_maximo"] * (
        1.0 - sd.exp(-1.0 * v["t_rendimiento"] * v["polinizacion_efectiva_total"]))
    return v
# Futuras iteraciones: modulo_quimico(model, p, v), modulo_espacial(...), modulo_economico(...)

In [ ]:
def simular(p=PARAMETROS, nombre="PolinizadoresBerries"):
    model = crear_modelo(p, nombre)
    v = modulo_polinizadores(model, p)
    v = modulo_agricola(model, p, v)
    bptk = BPTK_Py.bptk(); bptk.register_model(model)
    df = bptk.plot_scenarios(scenarios="base",
                             scenario_managers="sm" + nombre.capitalize(),  # regla "sm" + una sola mayúscula
                             equations=VARIABLES_SALIDA, return_df=True).reset_index()
    df["trimestre"] = (df["t"] / p["dt"]).round().astype(int)
    return df